# 라벨 EDA — deeplearning_labeling.zip 스트리밍 분석
압축 해제 없이 zip 직접 읽기

**실행 순서**
1. 환경 설정 (Drive 마운트 또는 파일 업로드)
2. ZIP 스트리밍 파싱 → CSV 저장
3. 작물 코드 매핑 확인
4. 작물×환경별 risk 분포
5. disease 분포 (개방형 일반화 설계용)
6. CSV 다운로드

## 1. 환경 설정

In [ ]:
# Google Drive 마운트 (Drive에 zip 올린 경우)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 경로 설정 ──────────────────────────────────────────────
# Drive에 올린 경우: '/content/drive/MyDrive/deeplearning_labeling.zip'
# Colab에 직접 업로드한 경우: '/content/deeplearning_labeling.zip'
ZIP_PATH   = '/content/drive/MyDrive/deeplearning_labeling.zip'
OUTPUT_CSV = '/content/label_eda_full.csv'
# ───────────────────────────────────────────────────────────

import zipfile, json, re
import pandas as pd
from tqdm.notebook import tqdm

# zip 열어서 전체 JSON 수 확인
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    total_json = sum(1 for e in zf.infolist() if e.filename.endswith('.json'))

print(f'ZIP 내 JSON 파일 수: {total_json:,}')

## 2. ZIP 스트리밍 파싱 → CSV 저장
> 증강(9.증강.zip) 포함 전체 파싱 — 기존 EDA(704,363장)와 동일 기준  
> `is_aug` 컬럼으로 원본/증강 구분  
> 약 20~40분 소요 예상

In [ ]:
records = []
skipped_error = 0

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    entries = [e for e in zf.infolist() if e.filename.endswith('.json')]

    for entry in tqdm(entries, desc='JSON 파싱'):
        path = entry.filename

        # ── 환경 ──
        if '071.시설' in path:
            env = '시설'
        elif '073.노지' in path:
            env = '노지'
        else:
            continue

        # ── 분할 ──
        if '1.Training' in path:
            split = 'train'
        elif '2.Validation' in path:
            split = 'val'
        else:
            continue

        # ── 작물 폴더명 (예: 01.가지) ──
        m = re.search(r'라벨링데이터/([^/]+)/', path)
        crop_folder = m.group(1) if m else 'unknown'

        # ── 원본/증강 구분 ──
        is_aug = '9.증강' in path

        # ── JSON 파싱 ──
        try:
            with zf.open(entry) as f:
                data = json.load(f)
        except Exception:
            skipped_error += 1
            continue

        ann = data.get('annotations', {})

        records.append({
            'env':         env,
            'split':       split,
            'crop_folder': crop_folder,
            'crop_code':   int(ann.get('crop',    -1)),
            'disease':     int(ann.get('disease', -1)),
            'risk':        int(ann.get('risk',    -1)),
            'grow':        int(ann.get('grow',    -1)),
            'is_aug':      is_aug,
        })

df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'\n총 레코드  : {len(df):,}')
print(f'  원본     : {(~df["is_aug"]).sum():,}')
print(f'  증강     : {df["is_aug"].sum():,}')
print(f'파싱 오류  : {skipped_error:,}')
print(f'저장 완료  : {OUTPUT_CSV}')
df.head()

## 3. 작물 코드 매핑 확인
폴더명(작물명) ↔ JSON crop_code 매핑 검증

In [ ]:
mapping = (
    df.groupby(['env', 'crop_folder', 'crop_code'])
    .size()
    .reset_index(name='count')
    .sort_values(['env', 'crop_folder'])
)
print('=== crop_folder → crop_code 매핑 ===')
print(mapping.to_string(index=False))

## 4. 작물×환경별 risk 분포
작물 선정 기준 결정용 — `min_class`가 낮은 그룹은 제외 또는 held-out 후보

In [ ]:
train_df = df[df['split'] == 'train'].copy()

pivot = (
    train_df.groupby(['env', 'crop_folder', 'risk'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: '정상(0)', 1: '초기(1)', 2: '중기(2)', 3: '말기(3)'})
)

# 없는 컬럼 보완
for col in ['정상(0)', '초기(1)', '중기(2)', '말기(3)']:
    if col not in pivot.columns:
        pivot[col] = 0

pivot['min_class'] = pivot[['정상(0)', '초기(1)', '중기(2)', '말기(3)']].min(axis=1)
pivot['total']     = pivot[['정상(0)', '초기(1)', '중기(2)', '말기(3)']].sum(axis=1)
pivot = pivot.reset_index().sort_values('min_class', ascending=False)

print('=== Train 기준 작물×환경별 risk 분포 (min_class 내림차순) ===')
print(pivot.to_string(index=False))

In [ ]:
# 포함/제외 기준 설정
THRESHOLD = 1000   # 필요하면 수정

include = pivot[pivot['min_class'] >= THRESHOLD][['env', 'crop_folder', 'min_class', 'total']]
exclude = pivot[pivot['min_class'] <  THRESHOLD][['env', 'crop_folder', 'min_class', 'total']]

print(f'\n[학습 포함 후보] min_class >= {THRESHOLD}')
print(include.to_string(index=False))

print(f'\n[held-out / 제외 후보] min_class < {THRESHOLD}')
print(exclude.to_string(index=False))

## 5. disease 분포 (개방형 일반화 설계용)
disease 코드별 risk 분포 → held-out disease 선정 기준

In [ ]:
# disease별 risk 분포
disease_pivot = (
    train_df[train_df['risk'] >= 0]
    .groupby(['env', 'crop_folder', 'disease', 'risk'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: '정상(0)', 1: '초기(1)', 2: '중기(2)', 3: '말기(3)'})
)

for col in ['정상(0)', '초기(1)', '중기(2)', '말기(3)']:
    if col not in disease_pivot.columns:
        disease_pivot[col] = 0

disease_pivot['min_class'] = disease_pivot[['정상(0)','초기(1)','중기(2)','말기(3)']].min(axis=1)
disease_pivot['total']     = disease_pivot[['정상(0)','초기(1)','중기(2)','말기(3)']].sum(axis=1)
disease_pivot = disease_pivot.reset_index().sort_values(['env', 'crop_folder', 'disease'])

print('=== disease별 risk 분포 ===')
print(disease_pivot.to_string(index=False))

In [ ]:
# 개방형 일반화 held-out 후보
# 조건: 4클래스 모두 존재(min_class > 0) + 충분한 샘플
DISEASE_THRESHOLD = 100

holdout_candidates = disease_pivot[
    (disease_pivot['min_class'] > 0) &
    (disease_pivot['min_class'] >= DISEASE_THRESHOLD)
][['env', 'crop_folder', 'disease', '정상(0)', '초기(1)', '중기(2)', '말기(3)', 'min_class', 'total']]

print(f'=== Held-out 가능 disease (4클래스 모두 >= {DISEASE_THRESHOLD}장) ===')
print(holdout_candidates.to_string(index=False))
print(f'\n총 {len(holdout_candidates)}개 disease가 held-out 후보')

## 6. 결과 CSV 다운로드

In [ ]:
from google.colab import files
files.download(OUTPUT_CSV)
print('다운로드 완료')